In [24]:
# AI-assisted
## Task 1 - Load and Inspect

import pandas as pd

df = pd.read_csv("bangalore_tech_salaries.csv")

print("Rows and Columns:", df.shape)
print("\nInformation")
print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())


Rows and Columns: (1015, 12)

Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Employee_ID     1015 non-null   object
 1   Role            1015 non-null   object
 2   years_exp       1015 non-null   int64 
 3   Current_CTC     1015 non-null   object
 4   Previous_CTC    815 non-null    object
 5   Company         1015 non-null   object
 6   company_TYPE    1015 non-null   object
 7   Skills          988 non-null    object
 8   Location        995 non-null    object
 9   Education_Tier  1015 non-null   object
 10  Joining_Year    1015 non-null   int64 
 11  Work_Mode       1015 non-null   object
dtypes: int64(2), object(10)
memory usage: 95.3+ KB
None

Missing Values
Employee_ID         0
Role                0
years_exp           0
Current_CTC         0
Previous_CTC      200
Company             0
company_TYPE        0
Skills  

In [25]:
## Task 2 - Data Cleaning
df.columns = df.columns.str.strip().str.lower()
def convert_ctc(value):
    if pd.isna(value):
        return value

    value = str(value).replace("₹","").replace("LPA","").replace(",","").strip()

    try:
        number = float(value)
    except:
        return None

    if number > 100000:
        number = number / 100000

    return number

df["current_ctc"] = df["current_ctc"].apply(convert_ctc)
df["previous_ctc"] = df["previous_ctc"].apply(convert_ctc)


df["company_type"] = df["company_type"].str.strip().str.title()


df["education_tier"] = df["education_tier"].replace({
"T1":"Tier 1",
"1":"Tier 1",
"Tier-1":"Tier 1",
"T2":"Tier 2",
"2":"Tier 2",
"Tier-2":"Tier 2",
"T3":"Tier 3",
"3":"Tier 3",
"Tier-3":"Tier 3"
})


print(df["role"].unique())


role_map = {
"Backend Engineer":"SDE Backend",
"Backend Dev":"SDE Backend",
"BE":"SDE Backend"
}

df["role_clean"] = df["role"].replace(role_map)

df = df.drop_duplicates()

print(df.dtypes)
print(df["role_clean"].value_counts())
print(df["company_type"].value_counts())
print(df["education_tier"].value_counts())


['DS' 'DevOps' 'SDE Backend' 'SDE Frontend' 'Product Lead'
 'Product Manager' 'Site Reliability Engineer' 'DA' 'UI/UX' 'SDE-Frontend'
 'UI Designer' 'ML Engineer' 'Frontend Dev' 'Data Scientist'
 'Full Stack Engineer' 'Designer' 'Product Designer' 'BA'
 'Frontend Engineer' 'BI Analyst' 'Backend Dev' 'Sr PM' 'FE' 'SDE-Backend'
 'Data Analyst' 'Business Analyst' 'Backend Developer'
 'Business Systems Analyst' 'Data Science Engineer' 'FullStack' 'BE'
 'Frontend Developer' 'Backend Engineer' 'UX Designer' 'Infra Engineer'
 'PM' 'SDE FS' 'DevOps Engineer' 'Analytics Engineer' 'Fullstack Dev'
 'SDE Full-Stack' 'SRE']
employee_id        object
role               object
years_exp           int64
current_ctc       float64
previous_ctc      float64
company            object
company_type       object
skills             object
location           object
education_tier     object
joining_year        int64
work_mode          object
role_clean         object
dtype: object
role_clean
SDE Backend       

In [26]:
# Create experience bands
# TASK -3 - Computations
bins=[0,2,4,6,100]
labels=["0-1","2-3","4-5","6+"]
df["exp_band"]=pd.cut(df["years_exp"],bins=bins,labels=labels,right=False)


In [27]:
# Q3.1
# Median, Mean, Min and Max salary by role
answer1=df.groupby("role_clean")["current_ctc"].agg(["median","mean","min","max"])
answer1=answer1.sort_values("median",ascending=False)
print(answer1)

                           median       mean   min       max
role_clean                                                  
Product Manager             31.60  34.385185  14.2  80.10000
Sr PM                       31.50  35.400000  10.8  65.50000
Product Lead                31.10  30.783870  12.0  55.10000
PM                          30.10  36.414286  13.8  78.50000
ML Engineer                 27.50  30.335484  10.0  64.50000
Data Science Engineer       25.65  28.356667  14.3  75.60000
FE                          25.30  24.108333  11.5  31.10000
SDE FS                      24.60  28.612500  11.5  70.90000
Fullstack Dev               24.55  21.821429   9.0  33.40000
Frontend Developer          24.30  23.604167  11.0  39.40000
SRE                         24.25  24.278571   8.8  49.50000
Data Scientist              23.90  25.746153  11.7  73.50000
Infra Engineer              23.45  22.610000   9.2  45.50000
SDE Full-Stack              23.05  28.568750  10.7  71.70000
Site Reliability Enginee

In [28]:
#Q3.2
backend=df[df["role_clean"]=="SDE Backend"]
result=backend.groupby("exp_band")["current_ctc"].median()
print(result)


exp_band
0-1    11.70
2-3    20.45
4-5    25.85
6+     40.20
Name: current_ctc, dtype: float64


/tmp/ipykernel_2727/2958182359.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  result=backend.groupby("exp_band")["current_ctc"].median()


In [29]:
#Q3.3
sde=df[df["role_clean"].isin(["SDE Backend","SDE Frontend","SDE Full Stack"])]

skills=["AWS","ML","System Design","Kubernetes"]

for skill in skills:
    with_skill=sde[sde["skills"].fillna("").str.contains(skill,case=False)]["current_ctc"].median()
    without_skill=sde[~sde["skills"].fillna("").str.contains(skill,case=False)]["current_ctc"].median()
    print(skill,with_skill,without_skill)


AWS 27.35 20.450000000000003
ML 25.9 21.9
System Design 30.15 21.1
Kubernetes 16.5 22.0


In [30]:
#Q3.4
backend=df[df["role_clean"]=="SDE Backend"]
company=backend.groupby("company_type")["current_ctc"].median()
print(company)


company_type
Early-Stage    18.60
Mid-Size       22.00
Mnc            20.45
Unicorn        27.35
Name: current_ctc, dtype: float64


In [31]:
#Q3.5
group_size=df.groupby(["role_clean","company_type","exp_band"])["current_ctc"].transform("count")
median=df.groupby(["role_clean","company_type","exp_band"])["current_ctc"].transform("median")

df["group_median"]=median
df["gap"]=df["current_ctc"]-df["group_median"]

underpaid=df[group_size>=10].sort_values("gap").head(10)

print(underpaid[["employee_id","role_clean","company_type","years_exp","current_ctc","group_median","gap"]])


    employee_id   role_clean company_type  years_exp  current_ctc  \
938     BLR0525  SDE Backend          Mnc          2         14.8   
184     BLR0918  SDE Backend          Mnc          2         18.0   
527     BLR0157  SDE Backend          Mnc          2         18.0   
958     BLR0390  SDE Backend          Mnc          2         19.3   
231     BLR0344  SDE Backend          Mnc          3         19.6   
870     BLR0160  SDE Backend          Mnc          3         20.3   
241     BLR0725  SDE Backend          Mnc          3         20.6   
44      BLR0793  SDE Backend          Mnc          3         20.9   
438     BLR0612  SDE Backend          Mnc          3         23.1   
185     BLR0934  SDE Backend          Mnc          3         23.3   

     group_median   gap  
938         20.45 -5.65  
184         20.45 -2.45  
527         20.45 -2.45  
958         20.45 -1.15  
231         20.45 -0.85  
870         20.45 -0.15  
241         20.45  0.15  
44          20.45  0.45  
438   

/tmp/ipykernel_2727/866588900.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_size=df.groupby(["role_clean","company_type","exp_band"])["current_ctc"].transform("count")
/tmp/ipykernel_2727/866588900.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  median=df.groupby(["role_clean","company_type","exp_band"])["current_ctc"].transform("median")


In [32]:
## Task 4 - Final Printed Report
print("="*60)
print("BANGALORE TECH SALARY DECODER")
print("="*60)
print("Dataset Size :",len(df))
print()

print("Median Salary by Role")
print(answer1["median"])

print()
print("Experience Band")
print(result)

print()
print("Company Type")
print(company)

print()
print("Top Underpaid Professionals")
print(underpaid[["employee_id","gap"]])
print("="*60)


BANGALORE TECH SALARY DECODER
Dataset Size : 1000

Median Salary by Role
role_clean
Product Manager              31.60
Sr PM                        31.50
Product Lead                 31.10
PM                           30.10
ML Engineer                  27.50
Data Science Engineer        25.65
FE                           25.30
SDE FS                       24.60
Fullstack Dev                24.55
Frontend Developer           24.30
SRE                          24.25
Data Scientist               23.90
Infra Engineer               23.45
SDE Full-Stack               23.05
Site Reliability Engineer    22.20
UI/UX                        22.00
SDE Backend                  21.95
Analytics Engineer           21.40
DevOps Engineer              21.15
DS                           21.00
Frontend Engineer            21.00
Business Systems Analyst     20.90
SDE-Backend                  20.60
BA                           20.50
Full Stack Engineer          20.50
Frontend Dev                 20.25
UI Des

TASK-5 INSIGHTS

INSIGHT 1: Employees with more experience earn significantly higher salaries. Professionals with 6+ years earn almost 3.4 times the salary of freshers.

INSIGHT 2: Unicorn companies offer the highest median salary (27.35 LPA), while Early-Stage startups pay the least (18.60 LPA).

INSIGHT 3: Among the analyzed skills, System Design has the highest salary impact, with professionals earning a median of 30.15 LPA compared to 21.10 LPA for those without it. Kubernetes appears to have a lower median salary in this dataset.